In [ ]:
import mdtraj as md
import numpy as np

In [ ]:
CORE_RESIDUES = (
    "resid 2 or resid 3 or resid 4 or "
    "resid 8 or resid 9 or resid 10 or "
    "resid 14 or resid 15 or resid 16 or "
    "resid 20 or resid 21 or resid 22"
)

In [ ]:
system_setup = {
    "blank": {
        "topo": "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/blank/na/step3_input.psf",
        "traj_list": [
            "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/blank/na/rep1/G4_ion_md_na.dcd",
            "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/blank/na/rep2/G4_ion_md_na.dcd",
            "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/blank/na/rep3/G4_ion_md_na.dcd",
        ],
        "label": "Blank Na+",
        "color": "green",
    }
}

In [ ]:
output_file = "rmsf_data_na_all_blank_c1.npz"

In [ ]:
results = {}
system_meta = {}
labels_cache = None

In [ ]:
print("Start RMSF calculation")

for sys_key, sys_info in system_setup.items():

    print(f"Processing system: {sys_key}")

    dcd_files = sys_info["traj_list"]
    rmsf_replicates = []

In [ ]:
ref_traj = md.load(
    dcd_files[0],
    top=sys_info["topo"],
    stride=100
)

ref_frame = ref_traj[0]

In [ ]:
align_selection = ref_frame.topology.select(
    f"({CORE_RESIDUES}) and (name P or name \"C1'\" or name \"C4'\")"
)

In [ ]:
if len(align_selection) == 0:
    print("[Warning] Core selection is empty. Use all P/C1'/C4' atoms instead.")

    align_selection = ref_frame.topology.select(
        "name P or name \"C1'\" or name \"C4'\""
    )

In [ ]:
calc_selection = ref_frame.topology.select(
    "name \"C1'\""
)

In [ ]:
if len(calc_selection) == 0:
    print("[Error] Cannot find C1' atoms.")
    print("Please check whether the atom name is C1' or C1*.")

In [ ]:
for rep_id, dcd in enumerate(dcd_files, start=1):

    print(f"Loading replica {rep_id}")

    traj = md.load(
        dcd,
        top=sys_info["topo"]
    )

    traj.superpose(
        ref_frame,
        atom_indices=align_selection
    )

    rmsf = md.rmsf(
        traj,
        traj,
        atom_indices=calc_selection
    )

    rmsf_replicates.append(rmsf)

    print(f"Replica {rep_id} finished")

In [ ]:
results[sys_key] = np.array(rmsf_replicates)

system_meta[sys_key] = {
    "label": sys_info["label"],
    "color": sys_info["color"],
}

In [ ]:
if labels_cache is None:

    selected_atoms = [
        traj.topology.atom(idx)
        for idx in calc_selection
    ]

    labels_cache = np.array([
        f"{atom.residue.name}{atom.residue.index + 1}"
        for atom in selected_atoms
    ])

In [ ]:
np.savez(
    output_file,
    results=results,
    labels=labels_cache,
    system_meta=system_meta
)

print(f"Finished. Output saved to: {output_file}")

In [ ]:
# ==================================================
# plot_rmsf_na_blank_avg_err.py
#
# Purpose:
#   Plot RMSF profile from NPZ file
#
# Input:
#   rmsf_data_na_all_blank_c1.npz
#
# Output:
#   RMSF Figure
#
# Data source:
#   Generated by calculate_rmsf_blank_na_c1.py
#
# Notes:
#   RMSF stored in NPZ is in nm
#   Plot is displayed in Å
# ==================================================

import numpy as np
import matplotlib.pyplot as plt


# ==================================================
# Part 1. User Settings
# ==================================================
#
# Most users only need to modify this section.
#
# ==================================================

# RMSF data file

data_file = "rmsf_data_na_all_blank_c1.npz"

# System to plot

target_key = "blank"

# G-tetrad core residues
# (0-based index)

core_indices = [
    1, 2, 3,
    7, 8, 9,
    13, 14, 15,
    19, 20, 21
]

# Fixed y-axis range
# Recommended for comparing multiple systems

y_max = 14.2


# ==================================================
# Part 2. Load RMSF Data
# ==================================================

try:

    loaded = np.load(
        data_file,
        allow_pickle=True
    )

    results = loaded["results"].item()

    labels = loaded["labels"]

    system_meta = loaded["system_meta"].item()

    data_loaded = True

    print(
        f"Successfully loaded: {data_file}"
    )

except FileNotFoundError:

    print(
        f"[ERROR] Cannot find {data_file}"
    )

    print(
        "Please run RMSF calculation first."
    )

    data_loaded = False


# ==================================================
# Part 3. Process RMSF Data
# ==================================================

if data_loaded:

    print(
        "\nGenerating RMSF plot ..."
    )

    if target_key not in results:

        print(
            f"[ERROR] Cannot find key: "
            f"{target_key}"
        )

    else:

        # ------------------------------------------
        # Load RMSF data
        # ------------------------------------------

        raw_data_nm = results[target_key]

        # Convert nm → Å

        raw_data_ang = raw_data_nm * 10

        # ------------------------------------------
        # Calculate statistics
        # ------------------------------------------

        rmsf_mean = np.mean(
            raw_data_ang,
            axis=0
        )

        rmsf_std = np.std(
            raw_data_ang,
            axis=0
        )

        # ------------------------------------------
        # Plot settings
        # ------------------------------------------

        system_label = system_meta[
            target_key
        ].get(
            "label",
            "System"
        )

        color = system_meta[
            target_key
        ].get(
            "color",
            "royalblue"
        )

        # Fix common typo

        if color == "seadgreen":
            color = "seagreen"


        # ==================================================
        # Part 4. Create Figure
        # ==================================================

        fig, ax = plt.subplots(
            figsize=(16, 8),
            dpi=150
        )

        num_residues = len(labels)

        x_positions = np.arange(
            num_residues
        )


        # ------------------------------------------
        # Draw G-tetrad core region
        # ------------------------------------------

        for idx in core_indices:

            ax.axvspan(
                idx - 0.5,
                idx + 0.5,
                color="gray",
                alpha=0.15,
                lw=0
            )

        ax.axvspan(
            -10,
            -10,
            color="gray",
            alpha=0.15,
            label="G-tetrad Core"
        )


        # ------------------------------------------
        # Mean RMSF
        # ------------------------------------------

        ax.plot(

            x_positions,
            rmsf_mean,

            marker="o",
            markersize=8,

            linewidth=3,

            linestyle="-",

            color=color,

            label="Mean RMSF (N=3)"

        )


        # ------------------------------------------
        # Standard deviation band
        # ------------------------------------------

        ax.fill_between(

            x_positions,

            rmsf_mean - rmsf_std,

            rmsf_mean + rmsf_std,

            color=color,

            alpha=0.3,

            lw=0,

            label=r"Std Dev ($\pm \sigma$)"

        )


        # ------------------------------------------
        # Title
        # ------------------------------------------

        ax.set_title(

            f"RMSF Analysis (C1' Atoms): "
            f"{system_label} (Na+)",

            fontsize=24,

            fontweight="bold"

        )


        # ------------------------------------------
        # Axis labels
        # ------------------------------------------

        ax.set_xlabel(

            "Residue Index (Sequential)",

            fontsize=22,

            fontweight="bold"

        )

        ax.set_ylabel(

            "RMSF (Å)",

            fontsize=22,

            fontweight="bold"

        )


        # ------------------------------------------
        # Tick labels
        # ------------------------------------------

        ax.set_xticks(
            x_positions
        )

        ax.set_xticklabels(

            labels,

            rotation=60,

            ha="right",

            fontsize=16

        )

        ax.tick_params(

            axis="both",

            which="major",

            labelsize=20

        )


        # ------------------------------------------
        # Axis range
        # ------------------------------------------

        ax.set_ylim(
            0,
            y_max
        )

        ax.set_xlim(
            -0.5,
            num_residues - 0.5
        )


        # ------------------------------------------
        # Grid and legend
        # ------------------------------------------

        ax.grid(

            axis="y",

            linestyle="--",

            alpha=0.7

        )

        ax.legend(

            fontsize=18,

            loc="upper right",

            framealpha=0.9

        )


        # ------------------------------------------
        # Show figure
        # ------------------------------------------

        plt.tight_layout()

        plt.show()

        print(
            "RMSF plot completed."
        )